# Low-Frequency Dosimetry with the SPFD Method

Induced electric field in a nine-layer conducting sphere exposed to a coil at 50 Hz,
computed with the finite integration technique. The benchmark and its analytical
reference solution are taken from Conchin Gubernati et al. (2022),
[doi:10.3390/app12136526](https://doi.org/10.3390/app12136526).

**Exposure.** A coil of radius 5 cm sits on the sphere axis, 13 cm above the centre,
carrying 1 kA at 50 Hz. The origin is the sphere centre; the coil axis is the U axis
of the FIT grid.

**Result.** On a 2 mm grid the per-layer maxima come out 12–33 % above the analytical
reference and the 99th percentiles 0–10 % above it. That gap is not a failure of the
method: it is the staircasing artifact the source paper set out to isolate, and this
benchmark is constructed so that it appears in isolation. The concluding section separates
it from the underlying accuracy.

## Why two solves rather than one

There are two independent reasons to split the problem, and they are worth keeping apart.
The first is physical: the reaction field of the induced currents is negligible, so the
source field can be computed without the tissue present. The second is numerical: the
monolithic operator has a scale separation of $10^{8}$–$10^{12}$ that no residual tolerance
can navigate. Either alone would justify the split; together they make it the standard
approach.

### The physical reason — the tissue is transparent

At 50 Hz the skin depth in the most conductive tissue (CSF, 2 S/m) is about 50 m, against a
phantom radius of 8 cm. The shielding parameter is therefore
$\mu_0\sigma\omega a^2 = 2(a/\delta)^2 \approx 5\times10^{-6}$: the induced currents screen
the applied field only at the parts-per-million level.

Curl Ampère's law, substitute Faraday, and use $\nabla\cdot\mathbf H = 0$ to get the
magnetic diffusion equation, then take it time-harmonic:

$$\nabla^2\mathbf H = \mu\sigma\,\partial_t\mathbf H
\qquad\longrightarrow\qquad
\nabla^2\mathbf H = \mathrm j\omega\mu\sigma\mathbf H .$$

Scaling lengths by the object size, $\mathbf x = a\tilde{\mathbf x}$, leaves

$$\tilde\nabla^2\mathbf H = \mathrm j\left(\omega\mu\sigma a^2\right)\mathbf H ,$$

so $\omega\mu\sigma a^2$ is the *only* dimensionless parameter in the governing equation.
Small values collapse it to $\tilde\nabla^2\mathbf H \approx 0$ — the conductor is invisible
to the field at leading order, with an $O(\omega\mu\sigma a^2)$ correction. Read as
timescales, $\mu\sigma a^2$ is the time for field to diffuse across the object and $1/\omega$
the time available, so a small ratio means diffusion keeps up and nothing is screened. The
identity $\mu\sigma\omega a^2 = 2(a/\delta)^2$ is then just algebra from
$\delta = \sqrt{2/(\omega\mu\sigma)}$.

For a homogeneous sphere in a uniform field the coefficient can be pinned down exactly:
$m = -2\pi a^3 H_0\left[1 - 3/x^2 + 3\cot x/x\right]$ with $x = ka$,
$k^2 = -\mathrm j\omega\mu\sigma$. Expanding $\cot x = 1/x - x/3 - x^3/45 - \cdots$ reduces
the bracket to $-x^2/15$, giving
$|\mathbf H_\text{reaction}|/|\mathbf H_0| \approx \omega\mu_0\sigma a^2/15
\approx 3\times10^{-7}$ for CSF. The bare group above therefore *overstates* the
perturbation by about 15× — the conservative direction. That coefficient is specific to a
uniform field and a homogeneous sphere; this phantom is layered and the coil field is
localized, so the geometric factor differs, but the scaling in $\omega\mu\sigma a^2$ is what
carries the argument.

### The numerical reason — the monolithic solve is ill-conditioned

The curl–curl block carries the reluctivity $\nu = 1/\mu_0 \approx 8\times10^5$, so on cells
of size $h$, with $\mathbf M_\nu \sim \nu/h$ and $\mathbf M_\sigma \sim \sigma h$,

$$\frac{\|\mathrm j\omega\mathbf M_\sigma\|}{\|\mathbf C^{\mathsf T}\mathbf M_\nu\mathbf C\|}
\;\sim\;\frac{\omega\sigma h}{\nu/h}\;=\;\mu_0\sigma\omega h^2
\;\approx\;3\times10^{-9}\quad(\text{CSF},\ h=2\ \text{mm}).$$

The tempting conclusion — "tiny perturbation, so drive the residual below it" — is the wrong
reading, because the two terms do not compete on the same subspace.
$\mathbf C^{\mathsf T}\mathbf M_\nu\mathbf C\,\mathbf G = \mathbf 0$ *identically*, so on the
gradients the curl–curl block contributes nothing and $\mathrm j\omega\mathbf M_\sigma$ acts
alone with eigenvalues $\sim\omega\sigma h$. Since the dosimetric correction
$\mathbf G\mathbf\Psi$ **is** a gradient, it lives exactly in the null space of the operator
carrying $1/\mu_0$: a Krylov residual reduction of $10^{-6}$ says almost nothing about it.
Condition numbers inside the conductor run from $3\times10^{8}$ (CSF) to $3\times10^{12}$
(skin), and in air, where $\sigma = 0$, the system is singular outright. This is the
low-frequency breakdown of the $\mathbf A$-formulation.

## The SPFD split

The SPFD scheme recovers the gradient part from its own equation instead:

1. **Source field** — magnetostatic curl–curl solve for the coil, tissue absent:
   $\mathbf{C}^{\mathsf T}\mathbf{M}_\nu\mathbf{C}\,\hat{\mathbf a}=\hat{\hat{\boldsymbol\jmath}}_{\mathrm s}$
   The quantity of interest here is the curl-dominated part of $\hat{\mathbf a}$, which is
   what a Krylov residual actually measures, so a modest tolerance suffices. The gradient
   part is left arbitrary on purpose — step 2 absorbs it (gauge invariance).
2. **Charge correction** — real SPD Poisson system on the conducting nodes only:
   $\mathbf{G}^{\mathsf T}\mathbf{M}_\sigma\mathbf{G}\,\mathbf\Psi
   =-\mathbf{G}^{\mathsf T}\mathbf{M}_\sigma\hat{\mathbf a}$
3. **Assembly** — $\hat{\mathbf e}=-\mathrm{j}\omega\left[\hat{\mathbf a}+\mathbf{G}\mathbf\Psi\right]$

$\mathbf{G}^{\mathsf T}\mathbf{M}_\sigma\mathbf{G}$ contains no $1/\mu_0$ anywhere — its
conditioning comes only from mesh size and the $10^4$ conductivity contrast — so neither
solve suffers the scale separation above.

$\mathbf\Psi$ is the time integral of the nodal potential, $\Psi=\varphi/(\mathrm{j}\omega)$.
Because $\hat{\mathbf a}$ is real, so is $\mathbf\Psi$: the Poisson solve runs in real
arithmetic, and $\omega$ and the coil current enter only as a final linear scaling.

## References

- T. W. Dawson, M. A. Stuchly, "An analytic solution for verification of computer models
  for low-frequency magnetic induction," *Radio Science*, vol. 32, no. 2, pp. 343–368, 1997.
  [doi:10.1029/96RS03791](https://doi.org/10.1029/96RS03791)
  *Analytic layered-sphere induction, written as a code verification benchmark. Solves the
  uniform-field case; the coil source used here needs the multipole extension.*
- T. W. Dawson, M. A. Stuchly, "Analytic Validation of a Three-Dimensional
  Scalar-Potential Finite-Difference Code for Low-Frequency Magnetic Induction,"
  *ACES Journal*, vol. 11, no. 3, pp. 72–81, November 1996. ISSN 1054-4887
  ([DTIC ADA319124](https://apps.dtic.mil/sti/pdfs/ADA319124.pdf); no DOI — the volume
  predates DOI assignment). *The original SPFD method.*
- A. Barchanski, M. Clemens, H. De Gersem, T. Weiland, "Efficient calculation of current
  densities in the human body induced by arbitrarily shaped, low-frequency magnetic field
  sources," *J. Comput. Phys.*, vol. 214, no. 1, pp. 81–95, 2006.
  [doi:10.1016/j.jcp.2005.09.009](https://doi.org/10.1016/j.jcp.2005.09.009)
  *Ex-SPFD: step 1 as a full FIT curl–curl solve, admitting arbitrarily shaped coils —
  the variant used here.*
- N. Haussmann, M. Zang, R. Mease, M. Clemens, B. Schmuelling, M. Bolten, "Towards
  real-time magnetic dosimetry simulations for inductive charging systems," *COMPEL*, 2021;
  preprint [arXiv:2010.12879](https://arxiv.org/abs/2010.12879).
  *Eqs. (1) and (3) give the $\mathbf\Psi$ form used above.*
- A. Conchin Gubernati, F. Freschi, L. Giaccone, R. Scorretti, "Analysis of Numerical
  Artifacts Using Tetrahedral Meshes in Low Frequency Numerical Dosimetry," *Appl. Sci.*,
  vol. 12, no. 13, 6526, 2022. [doi:10.3390/app12136526](https://doi.org/10.3390/app12136526)
  *Benchmark geometry and analytical reference values.*

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, "..", "..")) 
Pkg.instantiate() 
using FITToolbox;
using Statistics
using LinearAlgebra
using IterativeSolvers
using SparseArrays

function get_tangential_boundary(config::FITDomain)
    diag_vec = ones(Float64, 3*config.Np)
    # zero out the tangential entries; the rest pass through unchanged
    diag_vec[get_all_tangential_boundary_indices(config)] .= 0.0
    return spdiagm(0 => diag_vec)
end

## Step 1a — grid, source, and the curl–curl operator

### Graded mesh

`block_edges` builds a coarse–fine–coarse cell sequence along one axis: `n_outer` padding
cells of size `outer_res`, then `n_inner` cells of `inner_res`, then padding again.

| Axis | Cells | Extent | Fine region |
|---|---|---|---|
| U | 30 + 150 + 30 = 210 | 1500 mm | 600 – 900 mm |
| V | 30 + 100 + 30 = 160 | 1400 mm | 600 – 800 mm |
| W | 30 + 100 + 30 = 160 | 1400 mm | 600 – 800 mm |

That is 5.38 M cells, ~5.47 M nodes, ~16.4 M edge unknowns. The fine block is at
2 mm, the ICNIRP-recommended voxel resolution; the 600 mm of 20 mm padding on every
side is what keeps the truncation boundary condition from contaminating the result.

Everything of interest must sit inside the fine block — the coil now, the sphere later:

| Object | Extent | Fine region | |
|---|---|---|---|
| Coil plane, U | 850 mm | 600 – 900 | ✔ |
| Coil loop, V/W | 650 – 750 mm | 600 – 800 | ✔ |
| Sphere, U | 640 – 800 mm | 600 – 900 | ✔ |
| Sphere, V/W | 620 – 780 mm | 600 – 800 | ✔ |

The sphere lies entirely within the **uniform** 2 mm block, which later makes cell counts
a valid proxy for volume in the percentile statistics.

### Tissue absent, deliberately

`create_domain` is called with `σ = 0` everywhere. This is the SPFD assumption made
concrete: the phantom does not exist yet and is only introduced after $\hat{\mathbf a}$
has been solved for. The shielding parameter of §"Why two solves" is what licenses this —
adding the tissue here would change the source field by parts per million.

### Source

A circular loop of radius 50 mm, axis along U (`X()`), centred on the V/W midplane at
U = 850 mm, scaled to 1 kA. The result is the facet-integrated source current
$\hat{\hat{\boldsymbol\jmath}}_{\mathrm s}$.

### Curl–curl operator

`get_tangential_boundary` zeroes the tangential flux components on the outer boundary, so
flux leaves the truncated domain only normally. Composing it with the primal curl gives
the boundary-corrected $\mathbf C$, and with the reluctivity matrix
$\mathbf M_\nu = \mathbf M_\nu^{\mathsf T} \succ 0$,

$$\mathbf A = \mathbf C^{\mathsf T}\mathbf M_\nu\mathbf C .$$

$\mathbf A$ is symmetric positive **semi**-definite and singular:
$\mathbf A\mathbf G = \mathbf 0$, so the formulation is ungauged and any gradient may be
added to the solution. That is intentional — see the gauge-invariance argument at the
solve cell.

In [ ]:
function block_edges(outer_size, inner_size, outer_res, inner_res)
    n_outer = round(Int, outer_size / outer_res)
    n_inner = round(Int, inner_size / inner_res)
    return vcat(fill(float(outer_res), n_outer),
                fill(float(inner_res), n_inner),
                fill(float(outer_res), n_outer))
end

Edges_U = block_edges(600, 300, 20, 2)
Edges_V = block_edges(600, 200, 20, 2)
Edges_W = block_edges(600, 200, 20, 2)

MyDomain = create_domain(Edges_U, Edges_V, Edges_W; units = "mm", σ = 0, ε_r = 1, μ_r = 1)

# The coil plane sits at 850 mm, inside the fine U region (600–900 mm). In V and
# W the loop is centred at 700 mm with a 50 mm radius, so it spans 650–750 mm
# against a fine region of 600–800 mm.
posCoilU = 850.0
wire = create_circular_loop_source(MyDomain, 50.0, posCoilU, sum(Edges_V)/2, sum(Edges_W)/2, X(); units = "mm")
J = 1000.0 .* wire

# Applying the curl operator onto the vector potential, returns the magnetic flux in FIT;
# get_tangential_boundary sets the tangential components of the flux to zero and the flux can only escape via normal component
C = get_tangential_boundary(MyDomain) * get_curl(MyDomain,Primal());
G   = get_gradient(MyDomain, Primal())

M_ν = get_reluctivity(MyDomain);

A = C' * M_ν * C;
C = nothing; M_ν = nothing; GC.gc();

## Step 1b — solve the curl–curl system

$$\mathbf C^{\mathsf T}\mathbf M_\nu\mathbf C\,\hat{\mathbf a}
= \hat{\hat{\boldsymbol\jmath}}_{\mathrm s}$$

~16.4 M unknowns, so this is the expensive cell.

### Solving a singular system on purpose

$\mathbf A$ is symmetric positive semi-definite with $\mathbf A\mathbf G = \mathbf 0$: every
gradient is in the null space, and no gauge has been imposed. CG still converges, because
the right-hand side is **consistent**. The loop source is divergence-free, so
$\hat{\hat{\boldsymbol\jmath}}_{\mathrm s} \perp \operatorname{null}(\mathbf A)$, and every
Krylov vector $\mathbf A^k\hat{\hat{\boldsymbol\jmath}}_{\mathrm s}$ stays in
$\operatorname{range}(\mathbf A)$. The iteration therefore never enters the null space:
from a zero initial guess it returns the minimum-norm solution, with
$\mathbf G^{\mathsf T}\hat{\mathbf a}$ down at the level of accumulated rounding —
$1\times10^{-12}$ relative here, after 2655 iterations.

That would be harmless even if it were not: step 2 absorbs any gradient exactly, so the
assembled $\hat{\mathbf e}$ is gauge-invariant. If $\hat{\mathbf a} \to \hat{\mathbf a} +
\mathbf G\chi$, then $\mathbf\Psi \to \mathbf\Psi - \chi$ and
$\hat{\mathbf a} + \mathbf G\mathbf\Psi$ is unchanged.

### Why `reltol = 1e-6` is enough

What this solve needs to deliver is the **curl-dominated** part of $\hat{\mathbf a}$ — the
part carrying the physical flux — and that is precisely what the residual norm measures.
The gradient part is left arbitrary by design. This is the opposite situation to the
monolithic eddy-current solve, where the quantity of interest sits in the null space and
the residual says nothing about it.

That the gradient is genuinely irrelevant here can be checked rather than argued. Solving
the same system with a Jacobi preconditioner leaves 24 % gradient content in
$\hat{\mathbf a}$ instead of $10^{-12}$, yet the per-layer field values below agree with
the unpreconditioned run to 0.2 % on the maxima and exactly on the percentiles. Jacobi also
takes about twice the wall time for fewer iterations, which is why it is not used.

### Reported diagnostics

`history.isconverged` and `history.iters` come from the solver; the printed true residual
$\|\mathbf A\hat{\mathbf a} - \hat{\hat{\boldsymbol\jmath}}_{\mathrm s}\| /
\|\hat{\hat{\boldsymbol\jmath}}_{\mathrm s}\|$ is recomputed independently, since CG's
internal recursively-updated residual can drift from the true one over thousands of
iterations. The two should agree to a few digits; a large gap indicates loss of
orthogonality and is worth investigating before trusting the result.

In [ ]:
# can take a long time
â, history = cg(A, J; reltol = 1e-6, maxiter = 5000, log = true)
println("Converged:     ", history.isconverged)
println("Iterations:    ", history.iters)
println("True residual: ", norm(A * â - J) / norm(J))

## Step 1c — introduce the phantom

The sphere is created **after** the source field has been solved for. That ordering is the
SPFD assumption made operational: $\hat{\mathbf a}$ was computed with $\sigma = 0$
everywhere, and the tissue now added would have changed it only at the parts-per-million
level, so it is introduced purely to define $\mathbf M_\sigma$ for step 2.

### Layers

Nine concentric shells, centred at U = 720 mm (130 mm below the coil plane) on the V/W
midplane. `radii[i]` is the **outer** radius of shell `i`, so the arrays read inside-out:

| Shell (mm) | Tissue | σ (S/m) | Analytical max \|E\| (mV/m) |
|---|---|---|---|
| 0 – 38 | Brain | 0.11 | 4.12 |
| 38 – 42 | Cerebrospinal fluid | 2.0 | 4.76 |
| 42 – 64 | Brain | 0.11 | 9.78 |
| 64 – 66 | Cerebrospinal fluid | 2.0 | 10.40 |
| 66 – 68 | Muscle | 0.34 | 11.11 |
| 68 – 72 | Skull | 0.02 | 12.63 |
| 72 – 74 | Muscle | 0.34 | 13.47 |
| 74 – 76 | Fat | 0.043 | 14.37 |
| 76 – 80 | Skin | 0.0002 | 16.38 |

Geometry, conductivities and reference values are Tables 1 and 2 of Conchin Gubernati et
al. (2022). The $10^4$ conductivity contrast between CSF and skin is the sole source of
ill-conditioning in the step 2 Poisson system.

### Why the loop counts down

`create_sphere!` fills a solid ball, not a shell, so each call overwrites the interior of
the previous one. Iterating `length(radii):-1:1` lays down the 80 mm skin ball first and
the 38 mm brain core last, leaving the nesting above. Counting **up** would bury every
layer under the outermost one and give a homogeneous 0.0002 S/m sphere.

### Two layers share a conductivity

σ = 0.11 appears twice and σ = 2.0 appears twice, so conductivity does not identify a
layer. The post-processing separates shells by **radius** instead.

### The exposure is axisymmetric

The coil is coaxial with the sphere. **In the Coulomb gauge** this makes
$\mathbf A = A_\varphi(\rho,z)\hat{\mathbf e}_\varphi$ purely azimuthal, so
$\nabla\cdot(\sigma\mathbf A) = \rho^{-1}\partial_\varphi(\sigma A_\varphi) = 0$: the
induced currents circulate within each shell, never crossing a conductivity jump and never
reaching the outer surface. No charge accumulates, the exact $\varphi$ is zero, and
$\mathbf E = -\mathrm j\omega\mathbf A$.

That last step is conditional on the gauge, and step 1 here is ungauged — it returns
$\hat{\mathbf a} = \hat{\mathbf a}_\perp + \mathbf G\chi$ for an unknown $\chi$, giving
$\mathbf\Psi = -\chi + \text{const}$, which is not zero. What survives regardless of gauge
is the physical statement: $\hat{\mathbf e} = -\mathrm j\omega[\hat{\mathbf a} +
\mathbf G\mathbf\Psi]$ is invariant under $\chi$, so **E** is azimuthal up to staircasing
however the gauge falls out.

Whether $\mathbf\Psi$ itself is small is a separate, measurable question. Plain CG from
$x_0 = 0$ keeps every iterate in $\operatorname{range}(\mathbf A) \perp
\operatorname{null}(\mathbf A)$ and so returns the minimum-norm solution with
$\mathbf G^{\mathsf T}\hat{\mathbf a} = 0$ — a discrete Coulomb gauge. Jacobi
preconditioning breaks that, since $\mathbf M^{-1}$ does not preserve
$\operatorname{range}(\mathbf A)$ and $\mathbf A$ annihilates whatever gradient content
leaks in. Check with `norm(G * Ψ) / norm(â)`: small means $\mathbf\Psi$ carries only
staircasing, order unity means it is dominated by gauge freedom.

The symmetry is deliberate on the part of the benchmark. With no current crossing
interfaces, the only source of numerical error left is staircasing of the curved
boundaries, which is what the paper set out to isolate. If the check above comes back
small, the discrete $\mathbf\Psi$ is nonzero only because the voxelised sphere is not
perfectly axisymmetric — making it the mechanism by which staircasing produces hot spots,
rather than a correction to them.

In [ ]:
analyticalSolution = [4.12,4.76,9.78,10.4,11.11,12.63,13.47,14.37,16.38]

radii          = [38, 42, 64, 66, 68, 72, 74, 76, 80]
conductivities = [0.11, 2.0, 0.11, 2.0, 0.34, 0.02, 0.34, 0.043, 0.0002]

for i in length(radii):-1:1
    create_sphere!(MyDomain, posCoilU - 130, sum(Edges_V)/2, sum(Edges_W)/2,
                   radii[i]; units = "mm", σ = conductivities[i], ε_r = 1.0, μ_r = 1.0)
end

## Step 2 — the SPFD Poisson system

Current continuity $\nabla\cdot(\sigma\mathbf E) = 0$ with
$\mathbf E = -\mathrm j\omega\mathbf A - \nabla\varphi$ gives

$$\mathbf{G}^{\mathsf T}\mathbf{M}_\sigma\mathbf{G}\,\mathbf\Psi
= -\,\mathbf{G}^{\mathsf T}\mathbf{M}_\sigma\,\hat{\mathbf a}$$

written for $\mathbf\Psi = \int\!\varphi\,\mathrm dt = \varphi/(\mathrm j\omega)$, the time
integral of the nodal potential — Eq. (3) of Haussmann et al. Since $\hat{\mathbf a}$ is
real, so is $\mathbf\Psi$: no $\omega$ appears anywhere in this cell and the solve runs in
real arithmetic. The frequency re-enters only at assembly.

**Do not add a $\mathrm j\omega$ to `rhs_poisson` without also changing the assembly.** The
two conventions in the literature are
$(\varphi$: RHS carries $-\mathrm j\omega$, assembly is
$-\mathrm j\omega\hat{\mathbf a} - \mathbf G\varphi)$ and
$(\mathbf\Psi$: RHS bare, assembly is
$-\mathrm j\omega[\hat{\mathbf a} + \mathbf G\mathbf\Psi])$. Mixing rows leaves the gradient
term wrong by a factor $\omega \approx 314$.

### Restriction to conducting nodes

In air $\sigma = 0$, so those rows and columns of
$\mathbf G^{\mathsf T}\mathbf M_\sigma\mathbf G$ vanish identically — the air region carries
no equation at all. Selecting nodes with a nonzero diagonal restricts the system to the
sphere: roughly 0.27 M unknowns out of 5.47 M, a 20× reduction, and the printed count
should land near $\tfrac{4}{3}\pi(80\,\text{mm})^3 / (2\,\text{mm})^3 \approx 268{,}000$.

The restricted matrix is still singular — one constant per connected conducting component —
but consistent, since the right-hand side is a discrete divergence and is orthogonal to
constants. CG handles it exactly as in step 1. Only differences of $\mathbf\Psi$ enter the
assembly, so the undetermined constant never reaches the result.

### Conditioning

$\mathbf G^{\mathsf T}\mathbf M_\sigma\mathbf G$ contains no $1/\mu_0$: its conditioning comes
from the mesh size and the $10^4$ CSF-to-skin conductivity contrast alone, not from the
$10^{8}$–$10^{12}$ scale separation that afflicts the monolithic formulation. It is real,
sparse and SPD, which is why algebraic multigrid does well on it — Haussmann et al. reach a
$10^{-12}$ residual reduction on 8.9 M DOFs in under a second on a GPU. Plain Jacobi-free
CG at `reltol = 1e-8` is adequate at this size.

### Scattering back

`Ψ` is allocated over all nodes and filled only on the conducting set, leaving zeros in
air. Those zeros are not physical potentials — they are simply outside the domain of the
equation — and they are never read, because $\mathbf M_\sigma$ annihilates every
non-conducting edge in what follows.

In [ ]:
M_σ = get_conductivity(MyDomain)
G   = get_gradient(MyDomain, Primal())

A_poisson   = G' * M_σ * G
rhs_poisson = -(G' * (M_σ * â))

# Only nodes touching a conducting cell are determined; the air region has no
# equation at all and would leave the solver wandering in its null space.
conducting = findall(!iszero, diag(A_poisson))
println("conducting nodes: ", length(conducting), " of ", MyDomain.Np)

Ψ_c, h = cg(A_poisson[conducting, conducting], rhs_poisson[conducting]; reltol = 1e-8, maxiter = 5000, log = true)

Ψ = zeros(MyDomain.Np)
Ψ[conducting] = Ψ_c;

## Step 3 — assemble the electric grid voltages

$$\hat{\mathbf e} = -\mathrm j\omega\left[\hat{\mathbf a} + \mathbf G\mathbf\Psi\right]$$

Eq. (1) of Haussmann et al. This is where $\omega$ finally enters: steps 1 and 2 were both
frequency-free, so the whole result scales linearly with frequency and with coil current,
and either can be changed by rescaling rather than re-solving.

The two terms are the induced field and the charge correction:

- $-\mathrm j\omega\hat{\mathbf a}$ — the raw induced field from the changing flux, present
  whether or not tissue is there.
- $-\mathrm j\omega\mathbf G\mathbf\Psi$ — the field of the charge that accumulates wherever
  $-\mathrm j\omega\hat{\mathbf a}$ alone would drive current across a conductivity jump or
  out through the skin. It redirects the current so that
  $\nabla\cdot(\sigma\mathbf E) = 0$ and $\mathbf J\cdot\mathbf n = 0$ at the surface.

In this coaxial benchmark the exact correction is zero by symmetry, so the discrete
$\mathbf G\mathbf\Psi$ carries only staircasing artifacts. In any non-symmetric exposure it
carries a large part of the physics.

**Sign and grouping.** The bracket is a sum, not a difference — the $-\mathrm j\omega$
factors out of both terms. With the $\varphi$ convention it would read
$-\mathrm j\omega\hat{\mathbf a} - \mathbf G\varphi$ instead; the forms are not
interchangeable.

**Phase.** Both $\hat{\mathbf a}$ and $\mathbf\Psi$ are real, so $\hat{\mathbf e}$ is purely
imaginary and every component shares one phase. The field is linearly polarized, and
$\sqrt{\sum_k|E_k|^2}$ is therefore the true peak amplitude — which is what the norm in the
post-processing cell computes. Had the two terms sat in quadrature, that norm would not
have been the peak of the polarization ellipse.

**Consistency check.** `norm(G' * (M_σ * ê))` should sit at the step 2 solver tolerance
relative to `ω * norm(G' * (M_σ * â))`, since that is precisely the equation $\mathbf\Psi$
was solved from. A ratio near 1 instead of near zero means the convention got mixed.

In [ ]:
ω = 2π * 50
ê = -im * ω * (â .+ G * Ψ)

# Current continuity: GᵀM_σ ê should vanish, since that is the equation Ψ was
# solved from. The reference is the same quantity before the correction, scaled
# by ω so the two are comparable — it is what the residual would be if Ψ were
# left out entirely.
residual  = norm(G' * (M_σ * ê))
reference = ω * norm(G' * (M_σ * â))

println("‖GᵀM_σ ê‖          = ", residual)
println("ω‖GᵀM_σ â‖ (no Ψ)  = ", reference)
println("ratio              = ", residual / reference,
        "   (should sit near the step 2 tolerance, 1e-8)")

## Post-processing — |E| per tissue layer

Each shell is compared against the analytical reference of Conchin Gubernati et al.,
reported as a maximum, a 99th percentile, and the ratio max/ref.

### Sampling at cell centres

`FITToolbox.interpolate` returns the three field components in V/m, dividing each
edge-integrated value by its primal edge length internally — a length that differs per
direction on a graded mesh. Sampling at cell centres also sidesteps a genuine ambiguity:
a cell has exactly one conductivity, whereas an edge lying on a tissue interface belongs
to neither layer. Cell centres therefore give an unambiguous layer assignment and a
consistent unit conversion.

### Shells identified by radius

σ = 0.11 appears twice and σ = 2.0 twice, so conductivity cannot identify a layer. A cell
belongs to shell $(r_\text{inner}, r_\text{outer}]$ if its centre radius from the sphere
centre falls in that half-open interval — half-open so that consecutive shells partition
the sphere without overlap or gaps. The loop walks the shells outward, carrying
`r_prev` forward.

Note that this radial test re-derives layer membership geometrically rather than reading
back what `create_sphere!` actually assigned. The two agree only if the sphere is centred
where this cell assumes; the `cu, cv, cw` values are recomputed from the same expressions
passed to `create_sphere!` to keep them in step.

### Maximum and 99th percentile

Voxelising a curved boundary produces staircasing artifacts that surface as isolated hot
spots, so the raw maximum overestimates the exposure. `layer_values` returns the whole
vector rather than just the maximum so a percentile can be taken; ICNIRP suggests the
99th, though some authors prefer the 99.9th because the 99th can under-report localized
exposure. At 1 mm the source paper found the voxel maximum overshooting by ~15%, the
99.9th percentile by 5%, and the 99th under-reporting by 2%, against 0.5% for a
tetrahedral mesh. **This runs at 2 mm**, so expect larger deviations — the max/ref column
is the one to watch.

The unweighted `quantile` treats every cell equally, which is a valid volume percentile
only because the sphere sits entirely inside the uniform 2 mm block. It would be wrong if
any shell extended into the graded region.

### The norm

`hypot(abs.(E)...)` computes $\sqrt{\sum_k|E_k|^2}$, which is the peak amplitude precisely
because $\hat{\mathbf e}$ came out with a single common phase (see the assembly cell). For
an elliptically polarized field this would need the semi-major axis of the polarization
ellipse instead.

### Cost

The loop walks all 5.4 M cells per shell, computes a radius for each, and interpolates only
at the ~30 k that fall inside — so the cost is dominated by the radius test, not by the
interpolation. Nine shells means 48 M radius evaluations against 270 k interpolate calls.
Restricting the `i, j, k` ranges to the sphere's bounding box would remove most of the
former; computing all shells in one pass and binning by radius would remove the ninefold
repetition as well.

The `"no cells"` branch guards against a shell too thin to contain any cell centre. At 2 mm
the 64–66 and 72–74 shells are only one cell thick, so this is a real possibility rather
than defensive padding — in this run they resolve, with 13 544 and 17 112 cells
respectively.

In [ ]:
# ===========================================================================
# Maximum |E| per tissue layer
#
# interpolate returns the three field components in V/m — it divides the
# edge-integrated value by the primal edge length internally, which on a graded
# mesh differs per direction. Sampling at cell centres rather than reading edges
# directly also avoids the question of which layer an edge on a boundary belongs
# to: a cell has one conductivity, an edge between two layers has neither.
#
# Two shells share σ = 0.11 and two share σ = 2.0, so the layers are separated by
# radius rather than by conductivity.
# ===========================================================================

Nu, Nv, Nw = MyDomain.Nu, MyDomain.Nv, MyDomain.Nw

# Sphere centre, in the same coordinates passed to create_sphere! (mm).
cu = posCoilU - 130
cv = sum(Edges_V) / 2
cw = sum(Edges_W) / 2

# Cell centres in mm.
uc = MyDomain.edges_u_center .* 1000
vc = MyDomain.edges_v_center .* 1000
wc = MyDomain.edges_w_center .* 1000

"""
    layer_values(r_inner, r_outer) -> Vector{Float64}

|E| at the centre of every cell whose radius lies in the shell between
`r_inner` and `r_outer` (mm). Returns the values rather than just the maximum,
so a percentile can be taken as well — dosimetry standards use the 99th rather
than the peak, precisely because staircasing produces isolated outliers.
"""
function layer_values(r_inner, r_outer)
    vals = Float64[]
    for k in 1:Nw-1, j in 1:Nv-1, i in 1:Nu-1
        r = hypot(uc[i] - cu, vc[j] - cv, wc[k] - cw)
        (r_inner < r <= r_outer) || continue
        E = FITToolbox.interpolate(MyDomain, PrimalEdge(), ê,
                                   uc[i], vc[j], wc[k]; units = "mm")
        push!(vals, hypot(abs.(E)...))
    end
    return vals
end

println(rpad("shell (mm)", 12), rpad("σ", 9), rpad("cells", 8),
        rpad("max", 10), rpad("p99", 10), rpad("ref", 8), "max/ref")

r_prev = 0.0
for (r, σ, ref) in zip(radii, conductivities, analyticalSolution)
    v = layer_values(r_prev, r) .* 1000        # mV/m
    if isempty(v)
        println(rpad("$(round(Int,r_prev))–$(round(Int,r))", 12), "no cells")
    else
        println(rpad("$(round(Int,r_prev))–$(round(Int,r))", 12),
                rpad(σ, 9), rpad(length(v), 8),
                rpad(round(maximum(v), digits=2), 10),
                rpad(round(quantile(v, 0.99), digits=2), 10),
                rpad(ref, 8),
                round(maximum(v) / ref, digits=3))
    end
    global r_prev = r
end

## Conclusion

### The correction is the artifact

This benchmark is built so that the exact scalar potential vanishes. The coil is coaxial
with the sphere, the conductivity is radially layered, and the induced currents therefore
circulate within each shell without ever crossing an interface: $\nabla\cdot(\sigma\mathbf
A) = 0$ and $\varphi \equiv 0$ in the continuum. Every part of the discrete
$\mathbf G\mathbf\Psi$ is consequently a numerical artifact, produced only because the
voxelised sphere is not axisymmetric.

Assembling with and without it separates the two error sources cleanly:

| assembly | max error | p99 error |
|---|---|---|
| $-\mathrm j\omega\hat{\mathbf a}$ (source field only) | 1.0 – 2.7 % | −4.6 – 0.6 % |
| $-\mathrm j\omega[\hat{\mathbf a} + \mathbf G\mathbf\Psi]$ (full SPFD) | 11.8 – 33.0 % | −0.2 – 10.1 % |

The first row is what the discretisation of the source field and the geometry can achieve
on this mesh: about 2 %. The second is what the charge correction adds when the charge it
represents does not physically exist. The correction lands exactly on the stairstepped
interfaces, which is where the layer maxima are taken, so the peaks inflate by an order of
magnitude more than the mean field does.

Note the asymmetry between the two statistics. The maxima degrade from 2 % to 33 %; the
percentiles from about 1 % to 7 %. That is the signature of an artifact concentrated in a
small number of cells — which is precisely why exposure guidelines prescribe a percentile
rather than a peak.

### What this does and does not say about SPFD

None of this is an argument against the method. In any asymmetric exposure — a real
anatomy, an off-axis coil, a source with no symmetry to exploit — currents do cross tissue
boundaries, charge does accumulate, and $\mathbf G\mathbf\Psi$ carries physics that
$-\mathrm j\omega\hat{\mathbf a}$ alone cannot represent. Without it the field would be
continuous across every conductivity jump, where the correct behaviour is a discontinuity
in the ratio $\sigma_1/\sigma_2$.

What the benchmark shows is that on a voxel mesh the correction is computed from a
staircased conductivity map, and inherits its errors. The scheme is solving the right
equation for the wrong geometry — faithfully, which is why the artifact is systematic
rather than random.

### Current continuity

The correction does satisfy the equation it was solved from. After assembly,
$\|\mathbf G^{\mathsf T}\mathbf M_\sigma\hat{\mathbf e}\|$ is $1\times10^{-8}$ of the same
quantity computed without the correction, matching the `reltol` requested of the step 2
Poisson solve. That also confirms the $\mathbf\Psi$ convention is consistent between the
right-hand side and the assembly — a factor-of-$\omega$ mismatch would have left the ratio
near unity.

### Where the remedy lies

Because the error originates in the geometric representation of the interfaces rather than
in the formulation, it is addressed by conformal material averaging or a body-fitted mesh
rather than by uniform refinement. Refining a stairstepped boundary resolves its 90° corners
more closely; it does not remove them. That is the conclusion of the source paper, reached
here from the opposite direction: this notebook reproduces the artifacts the paper set out
to eliminate, and the two-row table above measures exactly what they cost.